Jakub Bagiński, 331357

Maciej Borkowski, 331667

# RAG – pytania i odpowiedzi w oparciu o bazę dokumentów

## Przygotowanie środowiska

In [ ]:
# ============================================================
# INSTALL
# ============================================================
!pip -q install llama-index
!pip -q install llama-index-vector-stores-chroma
!pip -q install llama-index-embeddings-huggingface
!pip -q install chromadb
!pip -q install sentence-transformers
!pip -q install transformers
!pip -q install accelerate
!pip -q install bitsandbytes
!pip -q install datasets
!pip -q install rank-bm25
!pip -q install spacy
!python -m spacy download en_core_web_sm
!pip install evaluate
!pip install rouge_score
!pip install sacrebleu

# ============================================================
# IMPORTS
# ============================================================
import os
import gc
import torch
import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline,
    BitsAndBytesConfig,
)
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext
import re
import spacy
import evaluate
from huggingface_hub import login
import transformers
transformers.logging.set_verbosity_error()

# ============================================================
# LOADING DATABASE
# ============================================================
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# ============================================================
# LOGIN TO HUGGINGFACE
# ============================================================
login(skip_if_logged_in=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Przygotowanie modeli i danych

### Konfiguracja

In [7]:
CONFIG = {
    # -----------------------------------------
    # DATA
    # -----------------------------------------
    "dataset_name": "rajpurkar/squad",
    "n_documents": 87599,
    # -----------------------------------------
    # CHUNKING
    # -----------------------------------------
    "chunk_size": 256,
    "chunk_overlap": 50,
    "len_short_chunk_to_remove": 40,
    "len_short_article_to_remove": 200,
    # -----------------------------------------
    # EMBEDDINGS
    # -----------------------------------------
    "embedding_model": "BAAI/bge-small-en-v1.5",
    # -----------------------------------------
    # RERANKER
    # -----------------------------------------
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    # -----------------------------------------
    # GENERATOR
    # -----------------------------------------
    "test_mode": False,
    "test_generator": "sshleifer/tiny-gpt2",
    "production_generator": "meta-llama/Meta-Llama-3-8B-Instruct",
    # "production_generator": "mistralai/Mistral-7B-Instruct-v0.2",
    # -----------------------------------------
    # RETRIEVAL
    # -----------------------------------------
    "dense_top_k": 20,
    "bm25_top_k": 20,
    # -----------------------------------------
    # RERANKING
    # -----------------------------------------
    "rerank_top_k": 10,
    # -----------------------------------------
    # OTHER
    # -----------------------------------------
    "recall_at_k": [1, 3, 5],
    "evaluation_n_samples": 100,
    "seed": 42,
}

### Przygotowanie danych

In [ ]:
# ============================================================
# LOAD SQUAD
# ============================================================
def create_dataset(shuffle=True):
    print("Loading SQuAD...")
    dataset = load_dataset(CONFIG["dataset_name"], split="train")
    if shuffle:
        dataset = dataset.shuffle(seed=CONFIG["seed"])
    return dataset.select(range(CONFIG["n_documents"]))

dataset = create_dataset()

# ============================================================
# BUILD DOCUMENTS
# ============================================================
def create_database(dataset):
    def deduplicate(docs, chunks):
        unique_texts = set()
        unique_documents = []
        unique_chunks = []

        for doc, chunk in zip(docs, chunks):
            if doc.text in unique_texts:
                continue

            unique_texts.add(doc.text)
            unique_documents.append(doc)
            unique_chunks.append(chunk)

        return unique_documents, unique_chunks
    print("Creating documents...")
    splitter = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"], chunk_overlap=CONFIG["chunk_overlap"]
    )
    documents = []
    chunks = []

    def clean_text(text: str):
        if not text:
            return ""
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    for row in tqdm(dataset):
        context = clean_text(row["context"])
        if len(context) < CONFIG["len_short_article_to_remove"]:
            continue
        source_doc = Document(
            text=f"Title: {row['title']}\n\n{context}",
            metadata={
                "question_id": row["id"],
                "title": row["title"],
                "question": row["question"],
                "gold_answer": row["answers"]["text"][0],
            },
        )

        nodes = splitter.get_nodes_from_documents([source_doc])
        for node in nodes:
            if len(node.text.split()) < CONFIG["len_short_chunk_to_remove"]:
                continue
            documents.append(node)
            chunks.append(
                {
                    "text": node.text,
                    "metadata": node.metadata,
                }
            )
    documents, chunks = deduplicate(documents, chunks)
    return documents, chunks

documents, chunks = create_database(dataset)

Loading SQuAD...
Creating documents...


  0%|          | 0/87599 [00:00<?, ?it/s]

Metadata length (248) is close to chunk size (256). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.


### Przygotowywanie komponetów RAG

In [ ]:
# ============================================================
# MODEL SELECTION
# ============================================================
GENERATOR_MODEL = (
    CONFIG["test_generator"] if CONFIG["test_mode"] else CONFIG["production_generator"]
)

print("Generator:", GENERATOR_MODEL)

# ============================================================
# BM25
# ============================================================
def create_lexical_model(chunks):
    def bm25_tokenizer(text):
        doc = nlp(text)
        return [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
            and len(token.text) > 2
        ]
    print("Building BM25...")
    tokenized_chunks = [bm25_tokenizer(chunk["text"]) for chunk in tqdm(chunks)]
    bm25 = BM25Okapi(tokenized_chunks)
    print("BM25 ready")
    return bm25, bm25_tokenizer

sparse_retriever, sparse_tokenizer = create_lexical_model(chunks)

# ============================================================
# EMBEDDING MODEL
# ============================================================
def create_embedding_model():
    print("Loading embeddings...")
    embedding_model = HuggingFaceEmbedding(model_name=CONFIG["embedding_model"])
    Settings.embed_model = embedding_model
    print("Embeddings loaded")
    return embedding_model

embedding_model = create_embedding_model()

# ============================================================
# CHROMA
# ============================================================
def build_chromadb(embedding_model):
    print("Building ChromaDB...")
    client = chromadb.PersistentClient(path="./chroma_db")
    # collection = client.get_or_create_collection("squad_rag")
    try:
        client.delete_collection("squad_rag")
    except Exception:
        pass
    collection = client.create_collection("squad_rag")
    vector_store = ChromaVectorStore(chroma_collection=collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, embed_model=embedding_model
    )
    dense_retriever = index.as_retriever(similarity_top_k=CONFIG["dense_top_k"])
    assert collection.count() == len(documents)
    print("Vector index ready")
    return dense_retriever

dense_retriever = build_chromadb(embedding_model)

# ============================================================
# RERANKER
# ============================================================
def create_reranker():
    print("Loading reranker...")
    reranker = CrossEncoder(CONFIG["reranker_model"])
    print("Reranker ready")
    return reranker

reranker = create_reranker()

# ============================================================
# GENERATOR
# ============================================================
def create_generator():
    print("Loading generator...")
    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
    if not CONFIG["test_mode"]:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            GENERATOR_MODEL, quantization_config=bnb_config, device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            GENERATOR_MODEL,
            torch_dtype=torch.float16,
            device_map="auto",
        )
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )
    print("Generator ready")
    return generator

generator = create_generator()

Generator: meta-llama/Meta-Llama-3-8B-Instruct
Building BM25...


  0%|          | 0/24651 [00:00<?, ?it/s]

BM25 ready
Loading embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Generator ready


### Konfiguracja generacji

In [ ]:
# ============================================================
# PROMPT TEMPLATE
# ============================================================
PROMPT_TEMPLATE = """
You are a question answering assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond exactly:

I could not find the answer.

Context:
{context}

Question:
{question}

Answer:
"""


# ============================================================
# GENERATION
# ============================================================
def generate_answer(question, context):
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    output = generator(prompt, max_new_tokens=128, do_sample=False)
    return output[0]["generated_text"]

### Definiowanie wariantów

In [ ]:
# ============================================================
# VARIANT A
# NO RAG
# ============================================================
def answer_A(question):
    prompt = f"""
Question:
{question}

Answer:
"""

    output = generator(prompt, max_new_tokens=128, do_sample=False)

    return output[0]["generated_text"]

# ============================================================
# VARIANT B
# BM25
# ============================================================
def retrieve_sparse(question, chunks, sparse_retriever, sparse_tokenizer):
    query_tokens = sparse_tokenizer(question)
    bm25_scores = sparse_retriever.get_scores(query_tokens)
    bm25_top_idx = np.argsort(bm25_scores)[::-1][: CONFIG["bm25_top_k"]]
    nodes = [chunks[i] for i in bm25_top_idx]
    return nodes

def answer_B(question):
    nodes = retrieve_sparse(question, chunks, sparse_retriever, sparse_tokenizer)
    docs = [node["text"] for node in nodes]
    context = "\n\n".join(docs)
    return generate_answer(question, context)

# ============================================================
# VARIANT C
# DENSE
# ============================================================
def retrieve_dense(question):
    nodes_with_scores = dense_retriever.retrieve(question)
    idx_nodes_with_scores = np.argsort([node.score for node in nodes_with_scores])[::-1][: CONFIG["dense_top_k"]]
    nodes_with_scores = [nodes_with_scores[i] for i in idx_nodes_with_scores]

    top_nodes = [{"text": node.text, "metadata": node.metadata} for node in nodes_with_scores]
    return top_nodes

def answer_C(question):
    nodes = retrieve_dense(question)
    docs = [node["text"] for node in nodes]
    context = "\n\n".join(docs)
    return generate_answer(question, context)

# ============================================================
# VARIANT D
# DENSE + RERANK
# ============================================================
def rerank(question, retrived_nodes):
    docs = [node["text"] for node in retrived_nodes]
    pairs = [(question, doc) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(retrived_nodes, scores), key=lambda x: x[1], reverse=True)
    nodes = [x[0] for x in ranked[: CONFIG["rerank_top_k"]]]
    return nodes

def retrieve_dense_rerank(question):
    dense_nodes = retrieve_dense(question)
    reranked_nodes = rerank(question, dense_nodes)
    return reranked_nodes

def answer_D(question):
    nodes = retrieve_dense_rerank(question)
    docs = [node["text"] for node in nodes]
    context = "\n\n".join(docs)
    return generate_answer(question, context)

# ============================================================
# MAIN API
# ============================================================
def ask(question, variant="D"):
    variant = variant.upper()
    if variant == "A":
        return answer_A(question)
    elif variant == "B":
        return answer_B(question)
    elif variant == "C":
        return answer_C(question)
    elif variant == "D":
        return answer_D(question)
    else:
        raise ValueError(f"Unknown variant: {variant}")

## Ewaluacja

In [ ]:
# ============================================================
# DEMO
# ============================================================
question = chunks[7]["metadata"]["question"]

print("\nQUESTION:")
print(question)

print("\nWariant A")
print(ask(question, variant="A"))
print("\nWariant B")
print(ask(question, variant="B"))
print("\nWariant C")
print(ask(question, variant="C"))
print("\nWariant D")
print(ask(question, variant="D"))


QUESTION:
What was Eisenhower's title after Germany's surrender?

Wariant A


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Question:
What was Eisenhower's title after Germany's surrender?

Answer:
Supreme Allied Commander of the Allied Forces in Europe. He held this position until the end of World War II. After the war, he was the 34th President of the United States.

Wariant B

You are a question answering assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
respond exactly:

I could not find the answer.

Context:
Title: Dwight_D._Eisenhower

The Germans launched a surprise counter offensive, in the Battle of the Bulge in December 1944, which the Allies turned back in early 1945 after Eisenhower repositioned his armies and improved weather allowed the Air Force to engage. German defenses continued to deteriorate on both the eastern front with the Soviets and the western front with the Allies. The British wanted Berlin, but Eisenhower decided it would be a military mistake for him to attack Berlin, and said orders to that effect would have to be explicit. The 

### Definowanie metryk ewaluacji

In [ ]:
rouge_metric = evaluate.load("rouge")
chrf_metric = evaluate.load("chrf")

# ============================================================
# ROGUE-L SCORE
# ============================================================
def rouge_l_score(reference, prediction):
    if not reference or not prediction:
        return 0.0

    result = rouge_metric.compute(
        predictions=[prediction],
        references=[reference],
        rouge_types=["rougeL"],
    )
    rouge_l = result.get("rougeL")
    return float(rouge_l or 0.0)

# ============================================================
# CHRF SCORE
# ============================================================
def chrf_score(reference, prediction):
    if not reference or not prediction:
        return 0.0

    result = chrf_metric.compute(
        predictions=[prediction],
        references=[reference],
    )
    return float(result.get("score", 0.0))

# ============================================================
# ANSWER QUALITY EVALUATION
# ============================================================
def evaluate_answer_quality(dataset_subset, variants=("A", "B", "C", "D")):
    rows = []
    for row in tqdm(dataset_subset, desc="Evaluating QA quality"):
        question = row["question"]
        gold_answer = row["answers"]["text"][0]
        for variant in variants:
            prediction = ask(question, variant=variant)
            prediction = prediction.partition("Answer:")[2].strip()
            rows.append(
                {
                    "variant": variant,
                    "rouge_l": rouge_l_score(gold_answer, prediction),
                    "chrf": chrf_score(gold_answer, prediction),
                }
            )
    df = pd.DataFrame(rows)
    return df.groupby("variant").mean().reset_index()

# ============================================================
# RECALL@K
# RETRIEVAL/RERANKING QUALITY EVALUATION
# ============================================================
def evaluate_retrieval_quality(dataset_subset, variants=("B", "C", "D")):
    def is_relevant_chunk(chunk, question_id):
        return chunk["metadata"].get("question_id") == question_id
    top_k_values = CONFIG["recall_at_k"]
    stats = {variant: {k: 0 for k in top_k_values} for variant in variants}
    total = len(dataset_subset)

    for row in tqdm(dataset_subset, desc="Evaluating retrieval"):
        question = row["question"]
        question_id = row["id"]

        for k in top_k_values:
            sparse_chunks = retrieve_sparse(
                question,
                chunks,
                sparse_retriever,
                sparse_tokenizer,
            )
            stats["B"][k] += any(
                is_relevant_chunk(node, question_id)
                for node in sparse_chunks[:k]
            )
            dense_chunks = retrieve_dense(question)
            stats["C"][k] += any(
                is_relevant_chunk(node, question_id)
                for node in dense_chunks[:k]
            )
            reranked_chunks = rerank(question, dense_chunks)
            stats["D"][k] += any(
                is_relevant_chunk(node, question_id)
                for node in reranked_chunks[:k]
            )

    rows = []
    for variant in variants:
        for k in top_k_values:
            rows.append(
                {
                    "variant": variant,
                    "k": k,
                    "recall": stats[variant][k] / total,
                }
            )
    return pd.DataFrame(rows)

### Eksperymenty

In [14]:
def run_experiments():
    n_samples = min(CONFIG["evaluation_n_samples"], len(dataset))
    data_subset = dataset.select(range(n_samples))
    print("Eksperyment 1 - Wpływ RAG na jakość odpowiedzi")
    qa_summary = evaluate_answer_quality(data_subset)
    print(qa_summary.to_string(index=False, float_format="%.4f"))

    print("Eksperyment 2 - Jakość wyszukiwania dokumentów")
    retrieval_summary = evaluate_retrieval_quality(data_subset)
    print(retrieval_summary.to_string(index=False, float_format="%.4f"))

run_experiments()

Eksperyment 1 - Wpływ RAG na jakość odpowiedzi


Evaluating QA quality:   0%|          | 0/100 [00:00<?, ?it/s]

variant  rouge_l    chrf
      A   0.0247  6.5278
      B   0.1521 20.1168
      C   0.1885 24.8143
      D   0.2712 32.2031
Eksperyment 2 - Jakość wyszukiwania dokumentów


Evaluating retrieval:   0%|          | 0/100 [00:00<?, ?it/s]

variant  k  recall
      B  1  0.6800
      B  3  0.8400
      B  5  0.8700
      C  1  0.9800
      C  3  0.9800
      C  5  0.9800
      D  1  0.8600
      D  3  0.9800
      D  5  0.9800
